In [2]:
import torch, os, cv2
import scipy.io as sio
import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont

In [3]:
data_root = "/home/jinhanz/cs/concreteness/data"

## Background Images

In [ ]:
imageinfo = pd.read_csv("/home/jinhanz/cs/concreteness/processing/stimuli_experiment/20250723_193742/image_info.csv")
stimuli_path = "/home/jinhanz/cs/concreteness/processing/stimuli_experiment/20250723_193742"

In [ ]:
save_dir_image = os.path.join(data_root, "image/images")
save_dir_whole_screen = os.path.join(data_root, "whole_screen/images")
save_dir_caption = os.path.join(data_root, "caption/images")

# caption bounding box
caption_xlo, caption_xhi, caption_ylo, caption_yhi = 0, 1280, 832, 1024

for b in range(1,5):

    b = str(b)
        
    for img_path in os.listdir(os.path.join(stimuli_path, b)):

        if 'mismatched' in img_path:
            continue

        condition = 'abstract' if 'abs' in img_path else 'concrete'

        imageinfo_row = imageinfo[imageinfo['trial_id'] == img_path].iloc[0]
        image_size = imageinfo_row["image_size"]

        img_id = imageinfo_row["image_id"]
        img_orig_path = f"/opt/jinhanz/data/mscoco/{img_id}"

        img_orig = Image.open(img_orig_path)

        os.makedirs(save_dir_image, exist_ok=True)
        os.makedirs(save_dir_whole_screen, exist_ok=True)
        os.makedirs(save_dir_caption, exist_ok=True)

        img_w, img_h = [int(coord) for coord in image_size.strip('[]').split(',')]
        img = img_orig.resize((img_w, img_h), Image.LANCZOS)
        img.save(os.path.join(save_dir_image, img_path))

        img_whole_screen = Image.open(os.path.join(stimuli_path, b, img_path))
        img_whole_screen.save(os.path.join(save_dir_whole_screen, img_path))

        img_caption = img_whole_screen.crop((caption_xlo, caption_ylo, caption_xhi, caption_yhi))
        img_caption.save(os.path.join(save_dir_caption, img_path))

## Fixation

In [ ]:
os.makedirs(os.path.join(data_root, 'image/fixations'), exist_ok=True)
os.makedirs(os.path.join(data_root, 'caption/fixations'), exist_ok=True)
os.makedirs(os.path.join(data_root, 'whole_screen/fixations'), exist_ok=True)

df_orig = pd.read_excel("/opt/jinhanz/results/2509_concreteness/dv_sessions/formal/fixations_report_250922.xlsx")
df_orig = df_orig[['CURRENT_FIX_X','CURRENT_FIX_Y','CURRENT_FIX_DURATION','imagename','RECORDING_SESSION_LABEL']]

df = df_orig[~df_orig['imagename'].str.contains('mismatched')]
df['image_num'] = df['imagename'].str.replace(r'(\.jpg)|(matched_(abs|con)_)|(COCO_val2014_)','', regex=True).astype(int)

grouped = df.groupby('imagename')

for name, group in grouped:

    # fixations_image
    imageinfo_row = imageinfo[imageinfo['trial_id'] == name].iloc[0]
    image_coordinates = imageinfo_row["image_coordinates"]

    x1, x2, y1, y2 = [int(coord) for coord in image_coordinates.strip('[]').split(',')]

    # Filter fixations within the bounding box
    y1 = 1024 - y1
    y2 = 1024 - y2
    y1, y2 = y2, y1
    
    filtered_img = group[
        (group['CURRENT_FIX_X'] >= x1) & (group['CURRENT_FIX_X'] <= x2) &
        (group['CURRENT_FIX_Y'] >= y1) & (group['CURRENT_FIX_Y'] <= y2)
    ]

    filtered_img['CURRENT_FIX_X'] = filtered_img['CURRENT_FIX_X'] - x1
    filtered_img['CURRENT_FIX_Y'] = filtered_img['CURRENT_FIX_Y'] - y1
    
    filtered_img.to_excel(os.path.join('/home/jinhanz/cs/concreteness/data/image/fixations', f"{name}.xlsx"),index=False)

    # fixations_caption
    filtered_caption = group[
        (group['CURRENT_FIX_X'] >= caption_xlo) & (group['CURRENT_FIX_X'] <= caption_xhi) &
        (group['CURRENT_FIX_Y'] >= caption_ylo) & (group['CURRENT_FIX_Y'] <= caption_yhi)
    ]

    filtered_caption['CURRENT_FIX_X'] = filtered_caption['CURRENT_FIX_X'] - caption_xlo
    filtered_caption['CURRENT_FIX_Y'] = filtered_caption['CURRENT_FIX_Y'] - caption_ylo

    filtered_caption.to_excel(os.path.join('/home/jinhanz/cs/concreteness/data/caption/fixations', f"{name}.xlsx"),index=False)

    # fixations_whole_screen
    filtered_whole_screen = group[
        (group['CURRENT_FIX_X'] >= 0) & (group['CURRENT_FIX_X'] <= 1280) &
        (group['CURRENT_FIX_Y'] >= 0) & (group['CURRENT_FIX_Y'] <= 1024)
    ]

    filtered_whole_screen['CURRENT_FIX_X'] = filtered_whole_screen['CURRENT_FIX_X'] - 0
    filtered_whole_screen['CURRENT_FIX_Y'] = filtered_whole_screen['CURRENT_FIX_Y'] - 0

    filtered_whole_screen.to_excel(os.path.join('/home/jinhanz/cs/concreteness/data/whole_screen/fixations', f"{name}.xlsx"),index=False)


## Heatmap Postprocessing & Save Human Visualizations

In [5]:
def convert_mat_to_pt(raw_image, mat_path, pth_path, visualization_path):
    mat = sio.loadmat(mat_path)
    map = mat['output_map_norm']

    map_tensor = torch.tensor(map, dtype=torch.float32)
    torch.save(map_tensor, pth_path)

    image = np.asarray(raw_image.copy())
    color = cv2.applyColorMap((map*255).astype(np.uint8), cv2.COLORMAP_JET) # cv2 to plt
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    c_ret = np.clip(image * (1 - 0.4) + color * 0.4, 0, 255).astype(np.uint8)
    heatmap = Image.fromarray(c_ret)
    heatmap.save(visualization_path)

for method in ['whole_screen','image','caption']: # 
    input_dir = os.path.join(data_root, f"{method}/mats")
    output_dir = os.path.join(data_root, f"{method}/pts")
    img_path = os.path.join(data_root, f"{method}/images")
    visualization_dir = os.path.join(data_root, f"{method}/visualizations/")

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(visualization_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        if filename.endswith('.mat'):
            mat_path = os.path.join(input_dir, filename)
            pt_path = os.path.join(output_dir, filename.replace('.mat', '.pt'))
            img_name = filename.replace('_GSmo_41.mat','')
            img = Image.open(os.path.join(img_path, img_name + '.jpg')).convert("RGB")
            visualization_path = os.path.join(visualization_dir, filename.replace('.mat', '.png'))
            convert_mat_to_pt(img, mat_path, pt_path, visualization_path)

## Save CLIP Visualizations

In [1]:
import json, random, csv, time, os, random
import numpy as np
from statistics import mean
from scipy.stats import ttest_rel, ttest_ind
import pandas as pd
from datetime import datetime
from tqdm import tqdm  # for progress bar
import matplotlib.pyplot as plt
from collections import defaultdict

# Load data
with open("../stimuli/coco_caption_concereteness_with_lemmatization.json") as f:
    data = json.load(f)

for i, img in enumerate(data):
    max_score, min_score = 0, 100
    max_id, min_id = 0, 0
    for c, cap in enumerate(img['caption_concreteness']):
        mean_score = mean(cap['concreteness'])
        data[i]['caption_concreteness'][c]['concrete_score'] = mean_score
        data[i]['caption_concreteness'][c]['abstract_score'] = mean_score
        if mean_score > max_score:
            max_score = mean_score
            max_id = cap['id']
        if mean_score < min_score:
            min_score = mean_score
            min_id = cap['id']
    data[i]['concrete_abstact_diff'] = {
        'value' : max_score - min_score,
        'ids': [max_id, min_id],
}

all_attributes = ["matching_score", "word_importances", "char_number", "word_number", "word_frequency", "emap_var", "abstract_score", "concrete_score"]
test_attributes = ["matching_score", "word_importances", "char_number", "word_number", "word_frequency", "emap_var"]

# Filter entries with both concrete and abstract ids
filtered_data = [item for item in data if len(set(item['concrete_abstact_diff']['ids'])) > 1]
filtered_data = sorted(filtered_data, key=lambda x: x['concrete_abstact_diff']['value'], reverse=True)

def extract_attr_lists(data_subset):
    ids = []
    abs_captions = []
    con_captions = []
    value_diffs = []
    abs_vals = {attr: [] for attr in all_attributes}
    con_vals = {attr: [] for attr in all_attributes}

    for entry in data_subset:
        ids.append(entry['image'])
        value_diffs.append(entry['concrete_abstact_diff']['value'])
        con_id, abs_id = entry['concrete_abstact_diff']['ids']
        con_cap = next(c for c in entry['caption_concreteness'] if c['id'] == con_id)
        abs_cap = next(c for c in entry['caption_concreteness'] if c['id'] == abs_id)

        abs_captions.append(abs_cap)
        con_captions.append(con_cap)

        for attr in ["concrete_score", "abstract_score"]:
            if attr in all_attributes:
                abs_vals[attr].append(abs_cap[attr])
                con_vals[attr].append(con_cap[attr])

        if 'char_number' in all_attributes:
            abs_vals['char_number'].append(len(abs_cap['caption']))
            con_vals['char_number'].append(len(con_cap['caption']))

        if 'word_number' in all_attributes:
            abs_vals['word_number'].append(len(abs_cap['caption'].split(' ')))
            con_vals['word_number'].append(len(con_cap['caption'].split(' ')))

        for attr in ["matching_score", "emap_var"]:
            if attr in all_attributes:
                abs_vals[attr].append(abs_cap[attr])
                con_vals[attr].append(con_cap[attr])

        for attr in ["word_importances", "word_frequency"]:
            if attr in all_attributes:
                abs_vals[attr].append(mean(abs_cap[attr]))
                con_vals[attr].append(mean(con_cap[attr]))

    return ids, value_diffs, abs_captions, con_captions, abs_vals, con_vals

ids_all, value_diffs_all, _, _, abs_vals_all, con_vals_all = extract_attr_lists(filtered_data)

experiment = '20250723_193742'

selected_indices = []
saved = pd.read_csv(f"/home/jinhanz/cs/concreteness/processing/stimuli_average/{experiment}/captions_and_attributes.csv")
saved_ids = saved['image_id'].tolist()
for e, entry in enumerate(filtered_data):
    if entry['image'] in saved_ids:
        selected_indices.append(e)

selected_indices.sort()
selected_entries = [filtered_data[idx] for idx in selected_indices]

ids_selected, value_diffs_selected, abs_captions_selected, con_captions_selected, abs_vals_selected, con_vals_selected = extract_attr_lists(selected_entries)

In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont
from datetime import datetime
import csv, cv2, torch
import torchvision.transforms as T

from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

def visualize(map, raw_image, resize):
    image = np.asarray(raw_image.copy())
    map = resize(map.unsqueeze(0))[0].cpu().numpy()
    color = cv2.applyColorMap((map*255).astype(np.uint8), cv2.COLORMAP_JET) # cv2 to plt
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    c_ret = np.clip(image * (1 - 0.4) + color * 0.4, 0, 255).astype(np.uint8)
    return Image.fromarray(c_ret)

# Prepare output directory
output_dir = '/home/jinhanz/cs/concreteness/data/human_and_clip/clip/image/'
os.makedirs(output_dir, exist_ok=True)
    
dict_out = {
    "image_id": ids_selected,
    "capion_concreteness_diff": value_diffs_selected,
    "abstract_caption":  [cap['caption'] for cap in abs_captions_selected],
    "concrete_caption": [cap['caption'] for cap in con_captions_selected],
}
for attr in all_attributes:
    dict_out["abs_"+ attr] = abs_vals_selected[attr]
    dict_out["con_"+ attr] = con_vals_selected[attr]
df_out = pd.DataFrame(dict_out)

for idx, (image_id, abs_cap, con_cap) in enumerate(zip(ids_selected, abs_captions_selected, con_captions_selected)):
    # Load image
    img_path = f"/home/jinhanz/cs/concreteness/data/image/images/matched_abs_{image_id.split('/')[-1]}"

    abs_emap = torch.load(f"../stimuli/emaps_resized/{image_id.split('_')[-1].split('.')[0]}_{abs_cap['id']}.pt", map_location="cpu")
    con_emap = torch.load(f"../stimuli/emaps_resized/{image_id.split('_')[-1].split('.')[0]}_{con_cap['id']}.pt", map_location="cpu")

    img = Image.open(img_path).convert("RGB")
    h, w = img.size
    resize = T.Resize((w,h))

    emaps =resize(torch.stack([abs_emap, con_emap], dim=0))

    abs_heatmap = visualize(emaps[0], img, resize)
    con_heatmap = visualize(emaps[1], img, resize)

    abs_heatmap.save(os.path.join(output_dir, f"matched_abs_{image_id.split('/')[-1]}.png"))
    con_heatmap.save(os.path.join(output_dir, f"matched_con_{image_id.split('/')[-1]}.png"))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import font_manager

def draw_highlighted_caption(caption, 
                             target_words,
                             concreteness_all, importance_all, 
                             start_pos=(60, 24*3), fontsize=14,
                             coloring=True,
                             base_color=(0, 0, 1),  # matplotlib uses 0–1 floats
                             highlight_color=(223/255, 245/255, 39/255),
                             save_path="highlighted_caption.png"):
    """
    Matplotlib version of highlighted text renderer.
    Creates its own figure/axes and saves result to file.
    """

    # Create fig/ax
    fig, ax = plt.subplots(figsize=(12.8, 1.92), dpi=129.94923858)
    ax.set_xlim(0, 1280)
    ax.set_ylim(0, 192)
    ax.axis('off')

    x, y = start_pos
    words = caption.strip('.,!?\'" ').split()

    renderer = fig.canvas.get_renderer()
    inv = ax.transData.inverted()

    for word, importance in zip(words, importance_all):
        word_clean = word.strip('.,!?\'"')
        vocab = word_clean.lower()

        # default black text
        color = (0, 0, 0, 1)

        if coloring:
            # background highlight rectangle
            highlight_alpha = importance
            bg_color = highlight_color + (highlight_alpha,)

            text = ax.text(x, y, word, fontsize=fontsize, color='black', va='top')
            bbox = text.get_window_extent(renderer=renderer)
            text.remove()

            bbox_data = bbox.transformed(inv)
            rect_width, rect_height = bbox_data.width, bbox_data.height

            rect = patches.Rectangle((x-0.05, y-rect_height*0.2),
                                     rect_width+0.1, rect_height*1.2,
                                     color=bg_color, zorder=1)
            ax.add_patch(rect)

            if vocab in target_words:
                idx = target_words.index(vocab)
                concreteness = concreteness_all[idx] / 5
                text_alpha = concreteness
                color = base_color + (text_alpha,)

        # draw text
        ax.text(x, y, word, fontsize=fontsize,
                color=color, zorder=2)

        # move x forward
        tmp = ax.text(x, y, word + " ", fontsize=fontsize, alpha=0)
        bbox = tmp.get_window_extent(renderer=renderer)
        tmp.remove()
        bbox_data = bbox.transformed(inv)
        word_width = bbox_data.width
        x += word_width

    # Save and close
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    return save_path

output_dir = '/home/jinhanz/cs/concreteness/data/human_and_clip/clip/caption/'
os.makedirs(output_dir, exist_ok=True)

for idx, (image_id, abs_cap, con_cap) in enumerate(zip(ids_selected, abs_captions_selected, con_captions_selected)):

    # Draw captions
    abs_target_words = abs_cap["words"]
    abs_conc_scores = abs_cap["concreteness"]
    abs_importance = abs_cap["word_importances"]

    con_target_words = con_cap["words"]
    con_conc_scores = con_cap["concreteness"]
    con_importance = con_cap["word_importances"]

    # Abstract caption
    draw_highlighted_caption(abs_cap['caption'],
                            abs_target_words,
                            abs_conc_scores, 
                            abs_importance, 
                         save_path=os.path.join(output_dir, f"matched_abs_{image_id.split('/')[-1]}.png"))

    draw_highlighted_caption(con_cap['caption'],
                            con_target_words,
                            con_conc_scores, 
                            con_importance, 
                         save_path=os.path.join(output_dir, f"matched_con_{image_id.split('/')[-1]}.png"))

## Compare Human and CLIP

In [42]:
clip_visualization_dir = "/home/jinhanz/cs/concreteness/processing/stimuli_average/20250723_193742"
human_visualization_image_dir = os.path.join(data_root, 'image/visualizations')
human_visualization_caption_dir = os.path.join(data_root, 'caption/visualizations')

def draw_heatmap(raw_image, pth_path):
    map = torch.load(pth_path).numpy()

    image = np.asarray(raw_image.copy())
    color = cv2.applyColorMap((map*255).astype(np.uint8), cv2.COLORMAP_JET) # cv2 to plt
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    c_ret = np.clip(image * (1 - 0.4) + color * 0.4, 0, 255).astype(np.uint8)
    heatmap = Image.fromarray(c_ret)
    return heatmap


for filename in os.listdir(clip_visualization_dir):
    if '.png' not in filename:
        continue
    clip_img_path = os.path.join(clip_visualization_dir, filename)

    img_id = '_'.join(filename.split('_')[1:]).replace('.jpg','').replace('.png','')

    # Human

    human_img_path = os.path.join(human_visualization_image_dir, "matched_CON_" + img_id + "_GSmo_41.png")
    human_caption_path = os.path.join(human_visualization_caption_dir, "matched_CON_" + img_id + "_GSmo_41.png")

    combined_img_path = os.path.join("/home/jinhanz/cs/concreteness/data/human_and_clip/comparison", filename)
    os.makedirs(os.path.dirname(combined_img_path), exist_ok=True)

    human_img_abs = Image.open(human_img_path.replace('CON','abs')).convert("RGB")
    human_caption_abs = Image.open(human_caption_path.replace('CON','abs')).convert("RGB")

    image_width, image_height = human_img_abs.size
    caption_width, caption_height = human_caption_abs.size
    width = max(image_width, caption_width)
    image_height = int(image_height / image_width * width)
    caption_height = int(caption_height / caption_width * width)

    human_img_abs = human_img_abs.resize((width, image_height), Image.LANCZOS)
    human_caption_abs = human_caption_abs.resize((width, caption_height), Image.LANCZOS)
    combined_abs = Image.new('RGB', (width, image_height + caption_height))
    combined_abs.paste(human_img_abs, (0, 0))
    combined_abs.paste(human_caption_abs, (0, image_height))

    human_img_con = Image.open(human_img_path.replace('CON','con')).convert("RGB")
    human_caption_con = Image.open(human_caption_path.replace('CON','con')).convert("RGB")
    human_img_con = human_img_con.resize((width, image_height), Image.LANCZOS)
    human_caption_con = human_caption_con.resize((width, caption_height), Image.LANCZOS)
    combined_con = Image.new('RGB', (width, image_height + caption_height))
    combined_con.paste(human_img_con, (0, 0))
    combined_con.paste(human_caption_con, (0, image_height))

    human_img = Image.new('RGB', (width * 2 + 80, image_height + caption_height), (255, 255, 255))
    human_img.paste(combined_abs, (20, 0))
    human_img.paste(combined_con, (width + 60, 0))
    human_img.save(combined_img_path.replace('comparison','human'))

    # CLIP

    clip_img_path = os.path.join(os.path.join(data_root, 'human_and_clip/clip/image'), "matched_CON_" + img_id + ".jpg.png")
    clip_caption_path = os.path.join(os.path.join(data_root, 'human_and_clip/clip/caption'), "matched_CON_" + img_id + ".jpg.png")

    clip_img_abs = Image.open(clip_img_path.replace('CON','abs')).convert("RGB")
    clip_caption_abs = Image.open(clip_caption_path.replace('CON','abs')).convert("RGB")

    clip_img_abs = clip_img_abs.resize((width, image_height), Image.LANCZOS)
    clip_caption_abs = clip_caption_abs.resize((width, caption_height), Image.LANCZOS)
    combined_abs = Image.new('RGB', (width, image_height + caption_height))
    combined_abs.paste(clip_img_abs, (0, 0))
    combined_abs.paste(clip_caption_abs, (0, image_height))

    clip_img_con = Image.open(clip_img_path.replace('CON','con')).convert("RGB")
    clip_caption_con = Image.open(clip_caption_path.replace('CON','con')).convert("RGB")
    clip_img_con = clip_img_con.resize((width, image_height), Image.LANCZOS)
    clip_caption_con = clip_caption_con.resize((width, caption_height), Image.LANCZOS)
    combined_con = Image.new('RGB', (width, image_height + caption_height))
    combined_con.paste(clip_img_con, (0, 0))
    combined_con.paste(clip_caption_con, (0, image_height))

    clip_img = Image.new('RGB', (width * 2 + 80, image_height + caption_height), (255, 255, 255))
    clip_img.paste(combined_abs, (20, 0))
    clip_img.paste(combined_con, (width + 60, 0))

    combined_img = Image.new('RGB', (width * 2 + 80, image_height * 2 + caption_height * 2))
    combined_img.paste(clip_img, (0, 0))
    combined_img.paste(human_img, (0, image_height + caption_height))

    combined_img.save(combined_img_path)